# Лабораторная работа №2 по BigDataSpark

В этой тетрадке описано моё решение данной лабораторную работу.

Выполнение данной работы я поделил на несколько частей:

1. Загружаю данные из CSV файлов в PostgreSQL таблицу `mock_data`.
2. Выделяю измерения `dim_customer`, `dim_seller`, `dim_product`, `dim_store`, `dim_supplier`.
3. Собираю таблицу фактов `fact_sales`.
4. Строю витрины `report_products`, `report_customers`, `report_time`, `report_stores`, `report_suppliers`, `report_quality`.
5. Проверяю итоговые количества строк и показываю примеры результатов.


## 1. Подключения и служебные функции

Сначала зададим все подключения и небольшие вспомогательные функции, чтобы дальше писать ETL уже без лишнего дублирования.


In [1]:
import pandas as pd
import psycopg2

from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

JARS = ",".join(
    [
        "/home/jovyan/jars/postgresql-42.7.3.jar",
        "/home/jovyan/jars/clickhouse-jdbc-0.6.0-all.jar",
    ]
)

PG_URL = "jdbc:postgresql://postgres:5432/postgres"
PG_PROPS = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver",
}

CH_URL = "jdbc:clickhouse://clickhouse:8123/lab2"
CH_PROPS = {
    "user": "admin",
    "password": "admin123",
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
}

UNKNOWN = "__UNKNOWN__"

STORE_COLS = [
    "store_name",
    "store_location",
    "store_city",
    "store_state",
    "store_country",
    "store_phone",
    "store_email",
]

SUPPLIER_COLS = [
    "supplier_name",
    "supplier_contact",
    "supplier_email",
    "supplier_phone",
    "supplier_address",
    "supplier_city",
    "supplier_country",
]

PG_EXPECTATIONS = {
    "mock_data": 10000,
    "dim_customer": 10000,
    "dim_seller": 10000,
    "dim_product": 10000,
    "fact_sales": 10000,
}

CLICKHOUSE_EXPECTATIONS = {
    "report_products": (1, 10000),
    "report_customers": (1, 10000),
    "report_time": 12,
    "report_stores": (1, 10000),
    "report_suppliers": (1, 10000),
    "report_quality": (1, 10000),
}


In [2]:
# Здесь поднимаю Spark и сразу подключаю JDBC-драйверы для PostgreSQL и ClickHouse.
spark = (
    SparkSession.builder.appName("BigDataSparkLab2")
    .config("spark.jars", JARS)
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.debug.maxToStringFields", "200")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


def read_pg(table_name):
    return spark.read.jdbc(PG_URL, table_name, properties=PG_PROPS)


def write_pg(df, table_name, mode="overwrite"):
    df.write.jdbc(PG_URL, table_name, mode=mode, properties=PG_PROPS)


def write_ch(df, table_name, order_by):
    (
        df.write.format("jdbc")
        .option("url", CH_URL)
        .option("dbtable", table_name)
        .option("user", CH_PROPS["user"])
        .option("password", CH_PROPS["password"])
        .option("driver", CH_PROPS["driver"])
        .option("isolationLevel", "NONE")
        .option("createTableOptions", f"ENGINE = MergeTree() ORDER BY ({order_by})")
        .mode("overwrite")
        .save()
    )


def read_ch(table_name):
    return (
        spark.read.format("jdbc")
        .option("url", CH_URL)
        .option("dbtable", table_name)
        .option("user", CH_PROPS["user"])
        .option("password", CH_PROPS["password"])
        .option("driver", CH_PROPS["driver"])
        .load()
    )


def pg_execute(sql):
    conn = psycopg2.connect(
        host="postgres",
        port=5432,
        dbname="postgres",
        user="postgres",
        password="postgres",
    )
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute(sql)
    conn.close()


def fill_unknown(column_name, alias=None):
    final_name = alias if alias else column_name
    return (
        F.when(F.col(column_name).isNull() | (F.trim(F.col(column_name)) == ""), F.lit(UNKNOWN))
        .otherwise(F.trim(F.col(column_name)))
        .alias(final_name)
    )


def global_order_window(*order_cols):
    return Window.partitionBy(F.lit(1)).orderBy(*order_cols)


def add_pk(table_name, column_name):
    pk_name = f"{table_name}_pkey"
    pg_execute(f'ALTER TABLE "{table_name}" ALTER COLUMN "{column_name}" SET NOT NULL;')
    pg_execute(
        f"""
        DO $$ BEGIN
            IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = '{pk_name}') THEN
                ALTER TABLE "{table_name}" ADD CONSTRAINT "{pk_name}" PRIMARY KEY ("{column_name}");
            END IF;
        END $$;
        """
    )


def add_fk(src_table, fk_name, src_col, tgt_table, tgt_col):
    pg_execute(
        f"""
        DO $$ BEGIN
            IF to_regclass('public.{src_table}') IS NOT NULL
               AND to_regclass('public.{tgt_table}') IS NOT NULL
               AND NOT EXISTS (
                   SELECT 1 FROM pg_constraint c
                   JOIN pg_class t ON t.oid = c.conrelid
                   WHERE c.conname = '{fk_name}' AND t.relname = '{src_table}'
               )
            THEN
                ALTER TABLE "{src_table}"
                ADD CONSTRAINT "{fk_name}"
                FOREIGN KEY ("{src_col}") REFERENCES "{tgt_table}"("{tgt_col}");
            END IF;
        END $$;
        """
    )


def drop_fks(table_name):
    pg_execute(
        f"""
        DO $$ DECLARE r record; BEGIN
            IF to_regclass('public.{table_name}') IS NOT NULL THEN
                FOR r IN
                    SELECT c.conname
                    FROM pg_constraint c
                    JOIN pg_class t ON t.oid = c.conrelid
                    WHERE c.contype = 'f' AND t.relname = '{table_name}'
                LOOP
                    EXECUTE format('ALTER TABLE %I DROP CONSTRAINT IF EXISTS %I', '{table_name}', r.conname);
                END LOOP;
            END IF;
        END $$;
        """
    )

print(f"Spark version: {spark.version}")


Spark version: 3.5.0


## 2. Проверка исходной таблицы

Перед построением схемы проверим, что таблица `mock_data` действительно содержит все 10000 строк и что структура выглядит нормально.


In [3]:
mock_data = read_pg("mock_data").cache()
mock_data_count = mock_data.count()

assert mock_data_count == PG_EXPECTATIONS["mock_data"], (
    f"Ожидалось {PG_EXPECTATIONS['mock_data']} строк в mock_data, а получилось {mock_data_count}"
)

print(f"Количество строк в mock_data: {mock_data_count}")
display(mock_data.limit(5).toPandas())


Количество строк в mock_data: 10000


,id,customer_first_name,customer_last_name,customer_age,customer_email,customer_country,customer_postal_code,customer_pet_type,customer_pet_name,customer_pet_breed,...,product_release_date,product_expiry_date,supplier_name,supplier_contact,supplier_email,supplier_phone,supplier_address,supplier_city,supplier_country,source_file
0,1,Barron,Rawlyns,61,bmassingham0@army.mil,China,None,cat,Priscella,Labrador Retriever,...,10/19/2011,10/21/2028,Tagcat,Bevan Massingham,bmassingham0@unblog.fr,914-877-7062,Suite 25,Kletek,China,/mock_data/MOCK_DATA.csv
1,2,Ham,Knowller,78,cscudder1@time.com,Poland,73-115,bird,Dalenna,Labrador Retriever,...,4/17/2019,2/29/2028,Livetube,Candide Scudder,cscudder1@sbwire.com,863-319-5653,18th Floor,Santa Cruz Balanyá,Poland,/mock_data/MOCK_DATA.csv
2,3,Farleigh,Langley,71,vhuxter2@fotki.com,Poland,56-160,bird,Aldridge,Parakeet,...,2/3/2010,9/3/2023,Photobug,Vinson Huxter,vhuxter2@slate.com,434-817-1275,Apt 96,Huangpu,Samoa,/mock_data/MOCK_DATA.csv
3,4,Shae,Debling,28,bbullier3@bravesites.com,Sri Lanka,81700,cat,Beverie,Labrador Retriever,...,8/22/2019,3/29/2026,Janyx,Bald Bullier,bbullier3@dot.gov,515-756-7392,12th Floor,Chengzhong,Indonesia,/mock_data/MOCK_DATA.csv
4,5,Lay,Twatt,25,sedgar4@smugmug.com,Botswana,None,bird,Sydelle,Siamese,...,12/9/2018,5/10/2030,Lazz,Sydelle Edgar,sedgar4@reverbnation.com,861-980-8811,Suite 79,Lanxi,China,/mock_data/MOCK_DATA.csv


## 3. Построение схемы звезда

Разобъём плоскую таблицу на измерения и факт. Для клиентов, продавцов и товаров беру ключи из исходных данных, а для магазинов и поставщиков генерирую суррогатные ключи после дедупликации.


In [4]:
# На всякий случай сначала убираю внешние ключи у fact_sales, если таблица уже была создана раньше.
drop_fks("fact_sales")

dim_customer = (
    mock_data.select(
        F.col("sale_customer_id").alias("customer_id"),
        F.trim(F.col("customer_first_name")).alias("customer_first_name"),
        F.trim(F.col("customer_last_name")).alias("customer_last_name"),
        F.col("customer_age"),
        F.trim(F.col("customer_email")).alias("customer_email"),
        F.trim(F.col("customer_country")).alias("customer_country"),
        F.trim(F.col("customer_postal_code")).alias("customer_postal_code"),
        F.trim(F.col("customer_pet_type")).alias("pet_type"),
        F.trim(F.col("customer_pet_name")).alias("pet_name"),
        F.trim(F.col("customer_pet_breed")).alias("pet_breed"),
        F.trim(F.col("pet_category")).alias("pet_category"),
    )
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)
write_pg(dim_customer, "dim_customer")

dim_seller = (
    mock_data.select(
        F.col("sale_seller_id").alias("seller_id"),
        F.trim(F.col("seller_first_name")).alias("seller_first_name"),
        F.trim(F.col("seller_last_name")).alias("seller_last_name"),
        F.trim(F.col("seller_email")).alias("seller_email"),
        F.trim(F.col("seller_country")).alias("seller_country"),
        F.trim(F.col("seller_postal_code")).alias("seller_postal_code"),
    )
    .filter(F.col("seller_id").isNotNull())
    .dropDuplicates(["seller_id"])
)
write_pg(dim_seller, "dim_seller")

print("dim_customer и dim_seller построены")


dim_customer и dim_seller построены


In [5]:
dim_product = (
    mock_data.select(
        F.col("sale_product_id").alias("product_id"),
        F.trim(F.col("product_name")).alias("product_name"),
        F.trim(F.col("product_category")).alias("product_category"),
        F.col("product_price"),
        F.col("product_quantity"),
        F.col("product_weight"),
        F.trim(F.col("product_color")).alias("product_color"),
        F.trim(F.col("product_size")).alias("product_size"),
        F.trim(F.col("product_brand")).alias("product_brand"),
        F.trim(F.col("product_material")).alias("product_material"),
        F.trim(F.col("product_description")).alias("product_description"),
        F.col("product_rating"),
        F.col("product_reviews"),
        F.when(
            F.col("product_release_date").rlike(r"^[0-9]{1,2}/[0-9]{1,2}/[0-9]{4}$"),
            F.to_date(F.col("product_release_date"), "M/d/yyyy"),
        )
        .otherwise(F.lit(None))
        .alias("product_release_date"),
        F.when(
            F.col("product_expiry_date").rlike(r"^[0-9]{1,2}/[0-9]{1,2}/[0-9]{4}$"),
            F.to_date(F.col("product_expiry_date"), "M/d/yyyy"),
        )
        .otherwise(F.lit(None))
        .alias("product_expiry_date"),
    )
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
)
write_pg(dim_product, "dim_product")

print("dim_product построена")


dim_product построена


In [6]:
store_unique = mock_data.select(*[fill_unknown(column_name) for column_name in STORE_COLS]).dropDuplicates(STORE_COLS)
dim_store = (
    store_unique.withColumn("store_id", F.row_number().over(global_order_window(*STORE_COLS)))
    .select("store_id", *STORE_COLS)
)
write_pg(dim_store, "dim_store")

supplier_unique = mock_data.select(*[fill_unknown(column_name) for column_name in SUPPLIER_COLS]).dropDuplicates(SUPPLIER_COLS)
dim_supplier = (
    supplier_unique.withColumn("supplier_id", F.row_number().over(global_order_window(*SUPPLIER_COLS)))
    .select("supplier_id", *SUPPLIER_COLS)
)
write_pg(dim_supplier, "dim_supplier")

print("dim_store и dim_supplier построены")


dim_store и dim_supplier построены


In [7]:
add_pk("dim_customer", "customer_id")
add_pk("dim_seller", "seller_id")
add_pk("dim_product", "product_id")
add_pk("dim_store", "store_id")
add_pk("dim_supplier", "supplier_id")

dim_store_db = read_pg("dim_store")
dim_supplier_db = read_pg("dim_supplier")

sales_base = mock_data.select(
    F.col("sale_customer_id"),
    F.col("sale_seller_id"),
    F.col("sale_product_id"),
    F.col("sale_quantity"),
    F.col("sale_total_price"),
    F.col("sale_date"),
    *[fill_unknown(column_name) for column_name in STORE_COLS],
    *[fill_unknown(column_name) for column_name in SUPPLIER_COLS],
)

fact_sales = (
    sales_base.join(dim_store_db.select("store_id", *STORE_COLS), STORE_COLS, "left")
    .join(dim_supplier_db.select("supplier_id", *SUPPLIER_COLS), SUPPLIER_COLS, "left")
    .select(
        F.to_date(F.col("sale_date"), "M/d/yyyy").alias("sale_date"),
        F.col("sale_customer_id").alias("customer_id"),
        F.col("sale_seller_id").alias("seller_id"),
        F.col("sale_product_id").alias("product_id"),
        F.col("store_id"),
        F.col("supplier_id"),
        F.col("sale_quantity"),
        F.col("sale_total_price"),
    )
    .withColumn(
        "sale_id",
        F.row_number().over(
            global_order_window(
                "sale_date",
                "customer_id",
                "seller_id",
                "product_id",
                "store_id",
                "supplier_id",
                "sale_total_price",
            )
        ),
    )
    .select(
        "sale_id",
        "sale_date",
        "customer_id",
        "seller_id",
        "product_id",
        "store_id",
        "supplier_id",
        "sale_quantity",
        "sale_total_price",
    )
)
write_pg(fact_sales, "fact_sales")

add_pk("fact_sales", "sale_id")

null_fk_rows = fact_sales.filter(
    F.col("customer_id").isNull()
    | F.col("seller_id").isNull()
    | F.col("product_id").isNull()
    | F.col("store_id").isNull()
    | F.col("supplier_id").isNull()
).count()
assert null_fk_rows == 0, f"В fact_sales нашлось {null_fk_rows} строк с NULL во внешних ключах"

add_fk("fact_sales", "fk_sales_customer", "customer_id", "dim_customer", "customer_id")
add_fk("fact_sales", "fk_sales_seller", "seller_id", "dim_seller", "seller_id")
add_fk("fact_sales", "fk_sales_product", "product_id", "dim_product", "product_id")
add_fk("fact_sales", "fk_sales_store", "store_id", "dim_store", "store_id")
add_fk("fact_sales", "fk_sales_supplier", "supplier_id", "dim_supplier", "supplier_id")

print("fact_sales построена")


fact_sales построена


In [8]:
star_counts = []
for table_name in ["dim_customer", "dim_seller", "dim_product", "dim_store", "dim_supplier", "fact_sales"]:
    star_counts.append((table_name, read_pg(table_name).count()))

display(pd.DataFrame(star_counts, columns=["table_name", "row_count"]))


,table_name,row_count
0,dim_customer,10000
1,dim_seller,10000
2,dim_product,10000
3,dim_store,10000
4,dim_supplier,10000
5,fact_sales,10000


## 4. Построение аналитических витрин

После того, как схема готова, прочитаем факт и измерения обратно из PostgreSQL и сделаем шесть витрин. Каждую витрину сразу запишем в ClickHouse.


In [9]:
fs = read_pg("fact_sales")
dp = read_pg("dim_product")
dcu = read_pg("dim_customer")
dst = read_pg("dim_store")
dsu = read_pg("dim_supplier")

print("Таблицы схемы звезда загружены для расчёта витрин")


Таблицы схемы звезда загружены для расчёта витрин


In [10]:
report_products = (
    fs.join(dp, "product_id", "left")
    .groupBy("product_id", "product_name", "product_category")
    .agg(
        F.sum("sale_total_price").alias("total_revenue"),
        F.sum("sale_quantity").alias("total_sales_quantity"),
        F.count("sale_id").alias("sales_count"),
        F.avg("product_rating").alias("avg_product_rating"),
        F.max("product_reviews").alias("product_reviews"),
    )
    .withColumn(
        "category_total_revenue",
        F.sum("total_revenue").over(Window.partitionBy("product_category")),
    )
    .withColumn(
        "product_rank_by_sales",
        F.row_number().over(
            global_order_window(F.col("total_sales_quantity").desc(), F.col("total_revenue").desc())
        ),
    )
    .withColumn(
        "is_top_10_product",
        F.when(F.col("product_rank_by_sales") <= 10, F.lit(1)).otherwise(F.lit(0)),
    )
)
write_ch(report_products, "report_products", "product_id")

print("report_products записана в ClickHouse")


report_products записана в ClickHouse


In [11]:
country_dist = dcu.groupBy("customer_country").agg(
    F.countDistinct("customer_id").alias("customers_in_country")
)

report_customers = (
    fs.join(dcu, "customer_id", "left")
    .groupBy("customer_id", "customer_first_name", "customer_last_name", "customer_country")
    .agg(
        F.sum("sale_total_price").alias("total_purchase_amount"),
        F.count("sale_id").alias("orders_count"),
        F.avg("sale_total_price").alias("avg_check"),
    )
    .join(country_dist, "customer_country", "left")
    .withColumn(
        "customer_rank_by_revenue",
        F.row_number().over(global_order_window(F.col("total_purchase_amount").desc())),
    )
    .withColumn(
        "is_top_10_customer",
        F.when(F.col("customer_rank_by_revenue") <= 10, F.lit(1)).otherwise(F.lit(0)),
    )
)
write_ch(report_customers, "report_customers", "customer_id")

print("report_customers записана в ClickHouse")


report_customers записана в ClickHouse


In [12]:
report_time = (
    fs.withColumn("sale_year", F.year("sale_date"))
    .withColumn("sale_month", F.month("sale_date"))
    .groupBy("sale_year", "sale_month")
    .agg(
        F.sum("sale_total_price").alias("monthly_revenue"),
        F.sum("sale_quantity").alias("monthly_sales_quantity"),
        F.count("sale_id").alias("orders_count"),
        F.avg("sale_total_price").alias("avg_order_size"),
    )
    .withColumn(
        "prev_month_revenue",
        F.coalesce(F.lag("monthly_revenue", 1).over(global_order_window("sale_year", "sale_month")), F.lit(0)),
    )
    .withColumn(
        "prev_year_same_month_revenue",
        F.coalesce(F.lag("monthly_revenue", 12).over(global_order_window("sale_year", "sale_month")), F.lit(0)),
    )
    .withColumn("revenue_delta_vs_prev_month", F.col("monthly_revenue") - F.col("prev_month_revenue"))
    .withColumn(
        "revenue_delta_vs_prev_year",
        F.col("monthly_revenue") - F.col("prev_year_same_month_revenue"),
    )
)
write_ch(report_time, "report_time", "sale_year, sale_month")

print("report_time записана в ClickHouse")


report_time записана в ClickHouse


In [13]:
report_stores = (
    fs.join(dst, "store_id", "left")
    .groupBy("store_id", "store_name", "store_city", "store_country")
    .agg(
        F.sum("sale_total_price").alias("total_revenue"),
        F.count("sale_id").alias("orders_count"),
        F.avg("sale_total_price").alias("avg_check"),
    )
    .withColumn(
        "store_rank_by_revenue",
        F.row_number().over(global_order_window(F.col("total_revenue").desc())),
    )
    .withColumn(
        "is_top_5_store",
        F.when(F.col("store_rank_by_revenue") <= 5, F.lit(1)).otherwise(F.lit(0)),
    )
    .withColumn("city_total_revenue", F.sum("total_revenue").over(Window.partitionBy("store_city")))
    .withColumn("country_total_revenue", F.sum("total_revenue").over(Window.partitionBy("store_country")))
)
write_ch(report_stores, "report_stores", "store_id")

print("report_stores записана в ClickHouse")


report_stores записана в ClickHouse


In [14]:
report_suppliers = (
    fs.join(dsu, "supplier_id", "left")
    .join(dp.select("product_id", "product_price"), "product_id", "left")
    .groupBy("supplier_id", "supplier_name", "supplier_country")
    .agg(
        F.sum("sale_total_price").alias("total_revenue"),
        F.count("sale_id").alias("orders_count"),
        F.avg("product_price").alias("avg_product_price"),
    )
    .withColumn(
        "supplier_rank_by_revenue",
        F.row_number().over(global_order_window(F.col("total_revenue").desc())),
    )
    .withColumn(
        "is_top_5_supplier",
        F.when(F.col("supplier_rank_by_revenue") <= 5, F.lit(1)).otherwise(F.lit(0)),
    )
    .withColumn("country_total_revenue", F.sum("total_revenue").over(Window.partitionBy("supplier_country")))
)
write_ch(report_suppliers, "report_suppliers", "supplier_id")

print("report_suppliers записана в ClickHouse")


report_suppliers записана в ClickHouse


In [15]:
report_quality = (
    fs.join(
        dp.select("product_id", "product_name", "product_rating", "product_reviews"),
        "product_id",
        "left",
    )
    .groupBy("product_id", "product_name")
    .agg(
        F.sum("sale_quantity").alias("total_sales_quantity"),
        F.sum("sale_total_price").alias("total_revenue"),
        F.avg("product_rating").alias("avg_product_rating"),
        F.max("product_reviews").alias("product_reviews"),
    )
    .withColumn(
        "highest_rating_rank",
        F.row_number().over(
            global_order_window(F.col("avg_product_rating").desc(), F.col("product_reviews").desc())
        ),
    )
    .withColumn(
        "lowest_rating_rank",
        F.row_number().over(
            global_order_window(F.col("avg_product_rating").asc(), F.col("product_reviews").desc())
        ),
    )
    .withColumn(
        "reviews_rank",
        F.row_number().over(global_order_window(F.col("product_reviews").desc())),
    )
    .withColumn(
        "is_highest_rated_product",
        F.when(F.col("highest_rating_rank") == 1, F.lit(1)).otherwise(F.lit(0)),
    )
    .withColumn(
        "is_lowest_rated_product",
        F.when(F.col("lowest_rating_rank") == 1, F.lit(1)).otherwise(F.lit(0)),
    )
    .withColumn(
        "is_most_reviewed_product",
        F.when(F.col("reviews_rank") == 1, F.lit(1)).otherwise(F.lit(0)),
    )
)

quality_corr = report_quality.agg(
    F.corr("avg_product_rating", "total_sales_quantity").alias("rating_sales_correlation")
)
report_quality = report_quality.crossJoin(quality_corr)
write_ch(report_quality, "report_quality", "product_id")

print("report_quality записана в ClickHouse")


report_quality записана в ClickHouse


## 5. Проверка результатов

Сверим количества строк в PostgreSQL и ClickHouse и отдельно посмотрим несколько строк из готовых витрин.


In [ ]:
validation_rows = []

for table_name, expected_count in PG_EXPECTATIONS.items():
    current_count = read_pg(table_name).count()
    assert current_count == expected_count, (
        f"Неожиданное количество строк в {table_name}: {current_count}, ожидалось {expected_count}"
    )
    validation_rows.append(("PostgreSQL", table_name, current_count, str(expected_count)))

for table_name in ["dim_store", "dim_supplier"]:
    current_count = read_pg(table_name).count()
    assert 1 <= current_count <= 10000, f"Неожиданное количество строк в {table_name}: {current_count}"
    validation_rows.append(("PostgreSQL", table_name, current_count, "1..10000"))

for table_name, expected in CLICKHOUSE_EXPECTATIONS.items():
    current_count = read_ch(table_name).count()
    if isinstance(expected, tuple):
        assert expected[0] <= current_count <= expected[1], (
            f"Неожиданное количество строк в {table_name}: {current_count}"
        )
        expected_label = f"{expected[0]}..{expected[1]}"
    else:
        assert current_count == expected, (
            f"Неожиданное количество строк в {table_name}: {current_count}, ожидалось {expected}"
        )
        expected_label = str(expected)
    validation_rows.append(("ClickHouse", table_name, current_count, expected_label))

validation_df = pd.DataFrame(
    validation_rows,
    columns=["storage", "table_name", "row_count", "expected"],
)

display(validation_df)

,storage,table_name,row_count,expected
0,PostgreSQL,mock_data,10000,10000
1,PostgreSQL,dim_customer,10000,10000
2,PostgreSQL,dim_seller,10000,10000
3,PostgreSQL,dim_product,10000,10000
4,PostgreSQL,fact_sales,10000,10000
5,PostgreSQL,dim_store,10000,1..10000
6,PostgreSQL,dim_supplier,10000,1..10000
7,ClickHouse,report_products,10000,1..10000
8,ClickHouse,report_customers,10000,1..10000
9,ClickHouse,report_time,12,12


Покажу несколько строк из итоговых витрин

In [ ]:
display(read_ch("report_products").orderBy(F.col("product_rank_by_sales").asc()).limit(5).toPandas())
display(read_ch("report_customers").orderBy(F.col("customer_rank_by_revenue").asc()).limit(5).toPandas())
display(read_ch("report_time").orderBy("sale_year", "sale_month").toPandas())
display(read_ch("report_stores").orderBy(F.col("store_rank_by_revenue").asc()).limit(5).toPandas())
display(read_ch("report_suppliers").orderBy(F.col("supplier_rank_by_revenue").asc()).limit(5).toPandas())
display(read_ch("report_quality").orderBy(F.col("highest_rating_rank").asc()).limit(5).toPandas())


Показываю несколько строк из итоговых витрин


,product_id,product_name,product_category,total_revenue,total_sales_quantity,sales_count,avg_product_rating,product_reviews,category_total_revenue,product_rank_by_sales,is_top_10_product
0,8616,Cat Toy,Toy,499.73,10,1,1.600000,290,868101.63,1,1
1,3550,Dog Food,Toy,499.59,10,1,1.900000,769,868101.63,2,1
2,1484,Bird Cage,Toy,498.96,10,1,3.300000,842,868101.63,3,1
3,7596,Dog Food,Food,498.35,10,1,2.300000,782,830632.55,4,1
4,9956,Dog Food,Food,497.25,10,1,4.100000,627,830632.55,5,1


,customer_country,customer_id,customer_first_name,customer_last_name,total_purchase_amount,orders_count,avg_check,customers_in_country,customer_rank_by_revenue,is_top_10_customer
0,Albania,4188,Gus,Hartshorn,499.85,1,499.850000,46,1,1
1,Portugal,6422,Hayes,McKain,499.80,1,499.800000,336,2,1
2,China,6361,Ava,Lomas,499.76,1,499.760000,1738,3,1
3,Indonesia,5923,Dawna,Impey,499.76,1,499.760000,1174,4,1
4,Poland,8616,Lavinia,Horsburgh,499.73,1,499.730000,332,5,1


,sale_year,sale_month,monthly_revenue,monthly_sales_quantity,orders_count,avg_order_size,prev_month_revenue,prev_year_same_month_revenue,revenue_delta_vs_prev_month,revenue_delta_vs_prev_year
0,2021,1,224158.54,4856,874,256.474302,0.00,0.00,224158.54,224158.54
1,2021,2,192348.31,4070,739,260.281881,224158.54,0.00,-31810.23,192348.31
2,2021,3,207282.20,4561,843,245.886358,192348.31,0.00,14933.89,207282.20
3,2021,4,206592.82,4564,837,246.825352,207282.20,0.00,-689.38,206592.82
4,2021,5,211764.86,4451,828,255.754662,206592.82,0.00,5172.04,211764.86
5,2021,6,215042.80,4438,822,261.609246,211764.86,0.00,3277.94,215042.80
6,2021,7,220496.51,4750,858,256.988939,215042.80,0.00,5453.71,220496.51
7,2021,8,221275.78,4818,897,246.684259,220496.51,0.00,779.27,221275.78
8,2021,9,210623.43,4507,839,251.041037,221275.78,0.00,-10652.35,210623.43
9,2021,10,228743.32,4976,892,256.438700,210623.43,0.00,18119.89,228743.32


,store_id,store_name,store_city,store_country,total_revenue,orders_count,avg_check,store_rank_by_revenue,is_top_5_store,city_total_revenue,country_total_revenue
0,1404,DabZ,Grekan,South Africa,499.85,1,499.850000,1,1,739.23,20072.29
1,7806,Thoughtblab,Fonte,Poland,499.80,1,499.800000,2,1,499.80,81313.13
2,1142,Camido,Longzhong,Sweden,499.76,1,499.760000,3,1,805.93,64148.78
3,2165,Edgeblab,Pesek,Indonesia,499.76,1,499.760000,4,1,499.76,279350.16
4,1266,Centizu,Tylicz,Poland,499.73,1,499.730000,5,1,641.52,81313.13


,supplier_id,supplier_name,supplier_country,total_revenue,orders_count,avg_product_price,supplier_rank_by_revenue,is_top_5_supplier,country_total_revenue
0,710,Brainverse,Ireland,499.85,1,31.420000,1,1,11636.18
1,3441,Jamia,Russia,499.80,1,6.640000,2,1,149206.75
2,1528,Demimbu,China,499.76,1,53.800000,3,1,492823.31
3,1948,Eabox,Portugal,499.76,1,80.910000,4,1,83210.60
4,912,Browsezoom,Argentina,499.73,1,58.240000,5,1,35606.32


,product_id,product_name,total_sales_quantity,total_revenue,avg_product_rating,product_reviews,highest_rating_rank,lowest_rating_rank,reviews_rank,is_highest_rated_product,is_lowest_rated_product,is_most_reviewed_product,rating_sales_correlation
0,2845,Dog Food,3,36.20,5.000000,998,1,9866,31,1,0,0,0.001005
1,7602,Dog Food,6,372.14,5.000000,991,2,9867,111,0,0,0,0.001005
2,2084,Dog Food,5,148.60,5.000000,987,3,9868,151,0,0,0,0.001005
3,5267,Bird Cage,8,228.16,5.000000,985,4,9869,165,0,0,0,0.001005
4,9523,Dog Food,1,410.77,5.000000,966,5,9870,389,0,0,0,0.001005


In [18]:
spark.stop()